# Phase 2 — SFT LoRA de Qwen3-1.7B-Base

Fine-tuning supervisé (SFT) de `Qwen/Qwen3-1.7B-Base` sur le dataset anonymisé de la phase 1, avec LoRA.

- **Prérequis** : GPU, fichiers `data/anonymized/sft_mediqal_mcqu_{train,validation,test}.anonymized.jsonl` produits par `data_prep_MediQAl`.
- **Étapes** : données → modèle → LoRA → entraînement → validation intermédiaire.
- **Suivi** : MLflow (`mlflow ui --backend-store-uri sqlite:///mlflow.db` depuis la racine du projet).

In [ ]:
# Installation (Colab ou machine GPU) : décommenter si besoin
# %pip install -q torch transformers peft trl datasets accelerate mlflow pandas

## 1. Configuration

In [1]:
import functools
import json
import re
from collections import Counter
from pathlib import Path

import mlflow
import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import SFTConfig, SFTTrainer

MODEL_NAME = "Qwen/Qwen3-1.7B-Base"
# Découpage d'origine de MediQAl, anonymisé en phase 1
DATA_FILES = {
    split: f"../data/anonymized/sft_mediqal_mcqu_{split}.anonymized.jsonl"
    for split in ("train", "validation", "test")
}

# Petit run pour valider la pipeline avant l'entraînement complet
SMALL_RUN = True

SEED = 42
RUN_NAME = f"sft-lora-r16-seed{SEED}" + ("-small" if SMALL_RUN else "")
OUTPUT_DIR = Path("../outputs") / RUN_NAME  # checkpoints et adaptateur final

set_seed(SEED)
if torch.cuda.is_available():
    device, bf16 = "cuda", torch.cuda.is_bf16_supported()  # bf16 absent sur T4 : fp16 à la place
elif torch.xpu.is_available():
    device, bf16 = "xpu", torch.xpu.is_bf16_supported()  # GPU Intel Arc
else:
    device, bf16 = "cpu", False
dtype = torch.bfloat16 if bf16 else torch.float16 if device != "cpu" else torch.float32
print(f"device={device}, dtype={dtype}")

if device == "xpu":
    # GPU Intel sous Windows : deux contournements
    # 1. Pas de compilateur C pour Triton : TRL calcule la loss en PyTorch pur
    import trl.kernels.chunked_logprob as chunked_logprob
    chunked_logprob._TRITON_AVAILABLE = False
    # 2. La norme des gradients en mode « foreach » fait planter le GPU : calcul tenseur par tenseur
    torch.nn.utils.clip_grad_norm_ = functools.partial(torch.nn.utils.clip_grad.clip_grad_norm_, foreach=False)

c:\Formation\Projet_14\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device=xpu, dtype=torch.bfloat16


## 2. Données : train / validation / test

In [2]:
# Format prompt / completion : la loss ne porte que sur la réponse de l'assistant
def to_prompt_completion(record):
    return {"prompt": record["messages"][:-1], "completion": record["messages"][-1:]}


splits = load_dataset("json", data_files=DATA_FILES)
splits = splits.map(to_prompt_completion, remove_columns=["messages"])
for name, ds in splits.items():
    print(f"{name}: {len(ds)} exemples")

# Petit run : sous-échantillon
if SMALL_RUN:
    for name, size in {"train": 200, "validation": 50, "test": 20}.items():
        splits[name] = splits[name].select(range(size))

train: 3935 exemples
validation: 999 exemples
test: 1996 exemples


## 3. Tokenizer et modèle

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Gabarit ChatML
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)

example = splits["train"][0]
print(tokenizer.apply_chat_template(example["prompt"] + example["completion"], tokenize=False))

<|im_start|>system
Tu es un assistant médical expert. Réponds de façon précise et concise.<|im_end|>
<|im_start|>user
Monsieur <PERSON> âgé de 37 ans, forestier de son métier, doit partir travailler pendant 2 ans au Gabon dans la région de Lambaréné. Monsieur <PERSON> a l'intention de se faire accompagner de son épouse âgée de 36 ans et de son fils âgé de 5 ans. Ils logeront à quelques kilomètres de la forêt mais Mr <PERSON>, de par son métier, passera la plupart de son temps au milieu des bois. Ils consultent 4 mois avant le départ pour se renseigner au sujet des vaccinations et de la chimioprophylaxie du paludisme. Vous lui expliquez les conseils d'hygiène, les éléments de prévention essentiels et les différents modes de contamination des maladies présentes dans la région

Au sujet des vaccinations :<|im_end|>
<|im_start|>assistant
Le vaccin contre la fièvre jaune est obligatoire<|im_end|>



In [4]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device)
print(f"{model.num_parameters() / 1e9:.2f} B paramètres")

Loading weights: 100%|██████████| 310/310 [00:02<00:00, 134.92it/s]


1.72 B paramètres


## 4. LoRA

In [5]:
# Seules de petites matrices de rang r sont entraînées : le modèle de base reste figé
lora_config = LoraConfig(
    r=16,                         # rang des matrices d'adaptation
    lora_alpha=32,                # facteur d'échelle (souvent 2 × r)
    lora_dropout=0.05,
    target_modules="all-linear",  # couches d'attention et MLP
    task_type="CAUSAL_LM",
)

## 5. Entraînement SFT

In [6]:
# Suivi MLflow : un run par entraînement, hyperparamètres et métriques enregistrés
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("qwen3-1.7b-sft-lora")
mlflow.start_run(run_name=RUN_NAME)
mlflow.log_params({"model": MODEL_NAME, "small_run": SMALL_RUN, "lora_r": lora_config.r, "lora_alpha": lora_config.lora_alpha, "lora_dropout": lora_config.lora_dropout})

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    run_name=RUN_NAME,
    seed=SEED,
    # Optimisation
    num_train_epochs=2,
    max_steps=30 if SMALL_RUN else -1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # batch effectif = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,  # 3 % des étapes en montée progressive du learning rate
    max_length=1024,
    gradient_checkpointing=True,  # moins de mémoire GPU, un peu plus lent
    bf16=bf16,
    fp16=dtype == torch.float16,
    # Évaluation et checkpoints
    eval_strategy="steps",
    eval_steps=10 if SMALL_RUN else 100,
    save_steps=10 if SMALL_RUN else 100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    # Logs
    logging_steps=5 if SMALL_RUN else 20,
    report_to="mlflow",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=splits["train"],
    eval_dataset=splits["validation"],
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


In [7]:
# Référence avant entraînement (adaptateur LoRA initialisé à zéro = modèle de base)
val_before = trainer.evaluate(metric_key_prefix="val_before")
val_before

Training Loss,Validation Loss,Step,Before Loss,Entropy,Mean Token Accuracy,Num Tokens
No log,No log,0,4.563701,2.835642,0.473971,0.000000


{'val_before_loss': 4.5637006759643555,
 'eval_entropy': 2.835641622543335,
 'eval_mean_token_accuracy': 0.4739705239023481,
 'eval_num_tokens': 0.0}

In [8]:
# Reprise après interruption : trainer.train(resume_from_checkpoint=True)
train_result = trainer.train()
train_result.metrics

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,3.081191,3.279757,2.875803,0.492129,44253.000000
20,2.519923,3.117466,2.954481,0.494380,87433.000000
30,2.460718,3.087798,2.873569,0.493400,130693.000000


{'train_runtime': 658.0131,
 'train_samples_per_second': 0.729,
 'train_steps_per_second': 0.046,
 'total_flos': 1594274044870656.0,
 'train_loss': 2.890885066986084,
 'epoch': 2.32}

In [9]:
# Sauvegarde de l'adaptateur LoRA (meilleur checkpoint) et de la configuration du run
FINAL_DIR = OUTPUT_DIR / "final"
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))

run_config = {
    "model": MODEL_NAME,
    "seed": SEED,
    "small_run": SMALL_RUN,
    "lora": {"r": lora_config.r, "alpha": lora_config.lora_alpha, "dropout": lora_config.lora_dropout},
    "data": {name: len(ds) for name, ds in splits.items()},
    "training_args": training_args.to_dict(),
}
(FINAL_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2, default=str), encoding="utf-8")

4579

## 6. Validation intermédiaire

1. **Loss et perplexité** sur val et test : elles doivent baisser par rapport au modèle de base.
2. **Génération** sur des exemples de test : comparaison modèle de base / modèle SFT.

In [10]:
val_after = trainer.evaluate(metric_key_prefix="val_after")
test_metrics = trainer.evaluate(splits["test"], metric_key_prefix="test")

losses = {
    "val avant SFT": val_before["val_before_loss"],
    "val après SFT": val_after["val_after_loss"],
    "test après SFT": test_metrics["test_loss"],
}
pd.DataFrame({"loss": losses, "perplexité": {k: torch.exp(torch.tensor(v)).item() for k, v in losses.items()}})

Training Loss,Validation Loss,Step,After Loss,Entropy,Mean Token Accuracy,Num Tokens
2.460718,No log,30,3.087798,2.873569,0.493400,130693.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 20/20 [00:00<00:00, 1105.35 examples/s]


Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
2.460718,3.344691,30,3.135587,0.458030,130693.000000


,loss,perplexité
val avant SFT,4.563701,95.937859
val après SFT,3.087798,21.928740
test après SFT,3.344691,28.351814


In [11]:
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")


@torch.no_grad()
def generate(prompt, max_new_tokens=128):
    # Décodage glouton : résultats reproductibles
    inputs = tokenizer.apply_chat_template(prompt, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
    output = trainer.model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        eos_token_id=[IM_END_ID, tokenizer.eos_token_id], pad_token_id=tokenizer.pad_token_id,
    )
    return tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def token_f1(prediction, reference):
    # Recouvrement de mots entre réponse générée et réponse attendue (0 à 1)
    pred, ref = re.findall(r"\w+", prediction.lower()), re.findall(r"\w+", reference.lower())
    common = sum((Counter(pred) & Counter(ref)).values())
    if common == 0:
        return 0.0
    precision, recall = common / len(pred), common / len(ref)
    return 2 * precision * recall / (precision + recall)

In [12]:
N_EVAL = 20 if SMALL_RUN else 100
trainer.model.eval()
trainer.model.config.use_cache = True  # accélère la génération

rows = []
for example in splits["test"].select(range(min(N_EVAL, len(splits["test"])))):
    reference = example["completion"][0]["content"]
    with trainer.model.disable_adapter():  # modèle de base, sans LoRA
        base_answer = generate(example["prompt"])
    sft_answer = generate(example["prompt"])
    rows.append({
        "question": example["prompt"][-1]["content"][-200:],
        "reference": reference,
        "base": base_answer,
        "sft": sft_answer,
        "f1_base": token_f1(base_answer, reference),
        "f1_sft": token_f1(sft_answer, reference),
    })

generations = pd.DataFrame(rows)
generations.to_csv(FINAL_DIR / "test_generations.csv", index=False)
generations[["f1_base", "f1_sft"]].mean()

f1_base    0.050983
f1_sft     0.036143
dtype: float64

In [13]:
# Quelques exemples à relire
pd.set_option("display.max_colwidth", 200)
generations[["question", "reference", "base", "sft"]].head(5)

,question,reference,base,sft
0,"abdominale montre un rein droit normal, à gauche une masse du pôle inférieur du rein, une lithiase vésicale et un résidu post mictionnel évalué à 100 cc\n\n(cochez la réponse juste) Il est logiqu...",Rechercher les métastases,Il est logique de :.RemoveEmptyEntries\nIl est logique de :.RemoveEmptyEntries\nIl est logique de :.RemoveEmptyEntries\nIl est logique de :.RemoveEmptyEntries\nIl est logique de :.RemoveEmptyEntri...,Laisser le patient en observation et de le faire revenir en cas de suspicion de grossissement du rein gauche. посколь\nمصالح\nL'adénome de la prostate est une cause de grossissement du rein gauche...
1,"ion du prurit l'amène à consulter. A l'examen, la peau, au niveau des mains et des bras présente des lésions de grattage et quelques lésions montrent des petites vésicules\n\nQuel est votre diagno...",Gale,"Votre patiente présente un syndrome de prurit cutané généralisé, probablement causé par une allergie cutanée. Les lésions de grattage et les petites vésicules sont des signes cliniques caractérist...",Eczéma de contact avec détergent الإلك\nمياه الإلك\nالبروبيوتيك الإلك\nالبروبيوتيك الإلك\nالبروبيوتيك الإلك\nالبروبيوتيك الإلك\nالبروبيوتيك الإلك\nالبروبيوتيك الإلك\nالبروبيوتيك الإلك\nالبروبيوتيك...
2,"personnes non obèses ont été suivis pendant 10 ans. Au cours des 10 ans de suivis 100 personnes ont développé un diabète, 80 sont des sujets obèses et 20 ne le sont pas.\n\nla valeur de cet indic...","0,04.","Pour étudier la relation entre l'obésité et l'apparition du diabète, un groupe de 1500 personnes obèses et un groupe de 1500 personnes non obèses ont été suivis pendant 10 ans. Au cours des 10 ans...",0.500.500.500.500.500.500.500.500.500.500.500.500.500.500.500.500.500.500.500.500.500.500.500.500.500.500.500.500.500.500.500.50
3,"ements. Les données d'examens sont en faveur d'une réaction méningée, ce que confirme la ponction lombaire\n\nParmi les propositions suivantes, laquelle(lesquelles) caractérise(nt) l'orchite ourli...",S'observe rarement avant la puberté,"A. Une inflammation de l'orchis\nB. Une inflammation de l'orchis et de la testicule\nC. Une inflammation de l'orchis, de la testicule et de la scrotum\nD. Une inflammation de l'orchis, de la testi...",Une inflammation de l'ovaire et de la testicule avec une douleur et une tuméfaction de ces deux organes. أنح\nمصال\nUne inflammation de l'ovaire et de la testicule avec une douleur et une tuméfact...
4,"obine : 8,8 g/l00 ml- Anticorps antinoyaux par immunofluorescence : 1 /2000ème de type homogèn\n\nLa survenue d'une phlébite du membre inférieur doit faire rechercher une anomalie biologique. Laqu...",Un anticoagulant de type anti-prothrombinase,La survenue d'une phlébite du membre inférieur doit faire rechercher une anomalie biologique. Laquelle ?,Une thrombophlébite de la veine cave supérieure. посколь\nCumhurbaşkan\nUne thrombophlébite de la veine cave inférieure. посколь\nCumhurbaşkan\nUne thrombophlébite de la veine pulmonaire. посколь\...


In [14]:
# Métriques finales dans MLflow, puis clôture du run
mlflow.log_metrics({
    "val_before_loss": val_before["val_before_loss"],
    "val_after_loss": val_after["val_after_loss"],
    "test_loss": test_metrics["test_loss"],
    "test_f1_base": generations["f1_base"].mean(),
    "test_f1_sft": generations["f1_sft"].mean(),
})
mlflow.end_run()

## Lecture des résultats

- `val après SFT` < `val avant SFT` : le modèle apprend le format et le domaine.
- `test` proche de `val` : pas de surapprentissage marqué.
- `f1_sft` > `f1_base` : les réponses se rapprochent des réponses attendues. Le F1 reste une mesure grossière (réponses libres, sans les propositions du QCM).

**Suite** : passer `SMALL_RUN = False` pour l'entraînement complet, puis DPO et contrôles de sécurité (hallucinations, recommandations dangereuses).